# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"

In [ ]:
# Please switch to the correct branch, fix or submit the changes, and then don't forget to re-run the pull or start a new session.
BRANCH = "dev/Stage-3"
!cd {PROJECT_ROOT} && git fetch origin && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
!cd {PROJECT_ROOT} && git branch --show-current && git log -1 --oneline


# To address the self-referential issue, if you have modified this notebook file, please open the temporary notebook and execute the following command:

# from google.colab import drive
# drive.mount('/content/drive')

# PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"
# BRANCH = "dev/Stage-3"

# !cd {PROJECT_ROOT} && git reset --hard && git fetch --all && git checkout {BRANCH} && git pull origin {BRANCH}
# !cd {PROJECT_ROOT} && git branch --show-current && git log -1 --oneline

In [ ]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en

---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [ ]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [ ]:
from Tools.download import download_mini

download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [ ]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [ ]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 60000,
    batch_size = 8,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "sgd",
    scheduler_name = "none",
    loss_name      = "qa_nll",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

---
## Section 4 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

---
## Section 5 — Stage 3: Experimental Investigation

Causal tracing adapted from Meng et al. (NeurIPS 2022) "Locating and Editing Factual Associations in GPT" to analyze QANet's internal mechanisms.

- **H1**: Component-level causal tracing (Conv vs Self-Attention vs FFN across Model Encoder layers)
- **H2**: Context vs Question dual-stream information flow & CQ Attention bottleneck
- **H3**: Pointer layer asymmetric design (M1/M2/M3 role differentiation)

In [ ]:
## Stage 3 — Setup: Load trained model for analysis

!pip install matplotlib seaborn -q

import argparse, json, os, random
import numpy as np
import torch
import torch.nn.functional as F
from Data import SQuADDataset, load_word_char_mats, load_train_dev_eval, make_loader
from Models import QANet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_PATH = "_model/model.pt"
DEV_NPZ = "_data/dev.npz"
DEV_EVAL_JSON = "_data/dev_eval.json"

# Load checkpoint and rebuild model
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
config = ckpt["config"]
model_args = argparse.Namespace(**config)

word_mat, char_mat = load_word_char_mats(model_args)
model = QANet(word_mat, char_mat, model_args).to(DEVICE)
state_key = "model_state" if "model_state" in ckpt else "model_state_dict"
model.load_state_dict(ckpt[state_key])
model.eval()

dataset = SQuADDataset(DEV_NPZ)
import ujson
with open(DEV_EVAL_JSON) as f:
    eval_file = ujson.load(f)

print(f"Model loaded on {DEVICE}. Dev set: {len(dataset)} samples.")
print(f"Checkpoint best F1: {ckpt.get('best_f1', '?')}, EM: {ckpt.get('best_em', '?')}")

### H1: Component-Level Causal Tracing in Model Encoder

**Hypothesis**: Conv, Self-Attention, and FFN play different causal roles across Model Encoder layers. Conv dominates shallow layers (local patterns), Attention grows in deep layers (global dependencies), FFN peaks mid-range.

**Method**: Corrupt context embedding → restore each sub-layer individually → measure Average Indirect Effect (AIE).

In [ ]:
from experiments.tracer import (
    qanet_forward, compute_span_prob, compute_start_prob, compute_end_prob,
    build_model_enc_specs, RestoreSpec,
)

NUM_SAMPLES_H1 = 200       # reduce if slow
NOISE_STD_SCALE = 3.0
NOISE_REPEATS = 3
MIN_CLEAN_PROB = 0.01
SEED = 42

random.seed(SEED); torch.manual_seed(SEED)
loader_h1 = make_loader(dataset, batch_size=1, shuffle=True)
specs_h1 = build_model_enc_specs()
spec_keys_h1 = [f"p{s.pass_idx}_b{s.block_idx}_{s.component}" for s in specs_h1]

h1_ie = {k: [] for k in spec_keys_h1}
h1_ie_p1 = {k: [] for k in spec_keys_h1}
h1_ie_p2 = {k: [] for k in spec_keys_h1}
h1_te_list = []
samples_used = 0

with torch.no_grad():
    for batch_idx, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(loader_h1):
        if samples_used >= NUM_SAMPLES_H1:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        y1, y2 = y1.to(DEVICE), y2.to(DEVICE)

        # Clean run
        p1_c, p2_c, clean_acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)
        prob_clean = compute_span_prob(p1_c, p2_c, y1, y2)
        if prob_clean.item() < MIN_CLEAN_PROB:
            continue

        rep_ie = {k: [] for k in spec_keys_h1}
        rep_ie_p1 = {k: [] for k in spec_keys_h1}
        rep_ie_p2 = {k: [] for k in spec_keys_h1}
        rep_te = []

        for rep in range(NOISE_REPEATS):
            ns = SEED + batch_idx * 100 + rep
            p1_x, p2_x, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid,
                                              corrupt_target="context", noise_std_scale=NOISE_STD_SCALE, noise_seed=ns)
            prob_corrupt = compute_span_prob(p1_x, p2_x, y1, y2)
            prob_p1_x = compute_start_prob(p1_x, y1)
            prob_p2_x = compute_end_prob(p2_x, y2)
            te = (prob_clean - prob_corrupt).item()
            rep_te.append(te)
            if abs(te) < 1e-6:
                continue

            for spec, key in zip(specs_h1, spec_keys_h1):
                p1_r, p2_r, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid,
                                                  corrupt_target="context", noise_std_scale=NOISE_STD_SCALE, noise_seed=ns,
                                                  clean_acts=clean_acts, restore_spec=spec)
                prob_r = compute_span_prob(p1_r, p2_r, y1, y2)
                rep_ie[key].append((prob_r - prob_corrupt).item())
                rep_ie_p1[key].append((compute_start_prob(p1_r, y1) - prob_p1_x).item())
                rep_ie_p2[key].append((compute_end_prob(p2_r, y2) - prob_p2_x).item())

        avg_te = np.mean(rep_te) if rep_te else 0.0
        h1_te_list.append(avg_te)
        for key in spec_keys_h1:
            if rep_ie[key]:
                h1_ie[key].append(np.mean(rep_ie[key]))
                h1_ie_p1[key].append(np.mean(rep_ie_p1[key]))
                h1_ie_p2[key].append(np.mean(rep_ie_p2[key]))
        samples_used += 1
        if samples_used % 50 == 0:
            print(f"  [{samples_used}/{NUM_SAMPLES_H1}] Avg TE so far: {np.mean(h1_te_list):.4f}")

# Aggregate
h1_results = {}
for key in spec_keys_h1:
    vs = np.array(h1_ie[key])
    n = len(vs)
    h1_results[key] = {
        "aie_span": float(vs.mean()) if n else 0.0,
        "aie_p1": float(np.mean(h1_ie_p1[key])) if n else 0.0,
        "aie_p2": float(np.mean(h1_ie_p2[key])) if n else 0.0,
        "ci95": float(vs.std() * 1.96 / np.sqrt(n)) if n > 1 else 0.0,
        "n": n,
    }

os.makedirs("experiments/results/H1", exist_ok=True)
with open("experiments/results/H1/h1_results.json", "w") as f:
    json.dump({"results": h1_results, "meta": {"num_samples": samples_used, "avg_te": float(np.mean(h1_te_list))}}, f, indent=2)

print(f"\nH1 done. {samples_used} samples, Avg TE = {np.mean(h1_te_list):.4f}")
print("Top 5 components by AIE (span):")
for k in sorted(h1_results, key=lambda k: h1_results[k]["aie_span"], reverse=True)[:5]:
    r = h1_results[k]
    print(f"  {k:<25s}  AIE={r['aie_span']:.4f} ± {r['ci95']:.4f}")

In [ ]:
## H1 — Visualization
import matplotlib.pyplot as plt

components = ["conv_0", "conv_1", "self_attn", "ffn"]
n_passes, n_blocks = 3, 7

# --- 1. Main heatmap ---
matrix = np.zeros((4, 21))
for p in range(3):
    for b in range(7):
        col = p * 7 + b
        for ci, comp in enumerate(components):
            matrix[ci, col] = h1_results.get(f"p{p}_b{b}_{comp}", {}).get("aie_span", 0)

fig, ax = plt.subplots(figsize=(18, 4))
vmax = max(abs(matrix.max()), abs(matrix.min()), 1e-6)
im = ax.imshow(matrix, cmap="RdYlBu_r", aspect="auto", vmin=-vmax*0.1, vmax=vmax)
ax.set_yticks(range(4)); ax.set_yticklabels(["Conv 0", "Conv 1", "Self-Attn", "FFN"])
ax.set_xticks(range(21)); ax.set_xticklabels([f"B{b}" for _ in range(3) for b in range(7)], fontsize=7)
for p in range(1, 3):
    ax.axvline(x=p*7-0.5, color="white", lw=2)
for p in range(3):
    ax.text(p*7+3, -0.8, f"Pass {p+1}→M{p+1}", ha="center", fontsize=10, fontweight="bold")
plt.colorbar(im, ax=ax, label="AIE (span)", shrink=0.8)
ax.set_title("H1: Component-Level Causal Tracing — QANet Model Encoder")
fig.tight_layout(); plt.show()

# --- 2. Aggregate bars ---
agg = {c: [] for c in components}
for key, val in h1_results.items():
    for c in components:
        if key.endswith(c):
            agg[c].append(val["aie_span"])
means = [np.mean(agg[c]) for c in components]
errs = [np.std(agg[c])/np.sqrt(len(agg[c]))*1.96 if agg[c] else 0 for c in components]

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(["Conv 0", "Conv 1", "Self-Attn", "FFN"], means, yerr=errs,
       color=["#2196F3","#42A5F5","#FF9800","#4CAF50"], capsize=5, edgecolor="black")
ax.set_ylabel("Average IE (span)"); ax.set_title("H1: Aggregate Causal Effect by Component")
ax.axhline(0, color="gray", ls="--", alpha=0.5); fig.tight_layout(); plt.show()

# --- 3. Start vs End difference ---
diff = np.zeros((4, 21))
for p in range(3):
    for b in range(7):
        col = p*7+b
        for ci, comp in enumerate(components):
            v = h1_results.get(f"p{p}_b{b}_{comp}", {})
            diff[ci, col] = v.get("aie_p1", 0) - v.get("aie_p2", 0)
fig, ax = plt.subplots(figsize=(18, 4))
vmax = max(abs(diff.max()), abs(diff.min()), 1e-6)
im = ax.imshow(diff, cmap="RdBu", aspect="auto", vmin=-vmax, vmax=vmax)
ax.set_yticks(range(4)); ax.set_yticklabels(["Conv 0","Conv 1","Self-Attn","FFN"])
ax.set_xticks(range(21)); ax.set_xticklabels([f"B{b}" for _ in range(3) for b in range(7)], fontsize=7)
for p in range(1,3): ax.axvline(x=p*7-0.5, color="black", lw=2)
plt.colorbar(im, ax=ax, label="AIE_p1 − AIE_p2  (red→start, blue→end)")
ax.set_title("H1: Start vs End Position Bias"); fig.tight_layout(); plt.show()

### H2: Context vs Question Dual-Stream Information Flow

**Hypothesis**: Question corruption is more destructive than Context corruption. CQ Attention is the critical information bottleneck (recovering its output restores >70% of TE).

**Method**: Three corruption conditions (C / Q / Both) × restore Embedding Encoder sub-layers and CQ Attention.

In [ ]:
from experiments.tracer import build_emb_enc_specs, build_fusion_specs

NUM_SAMPLES_H2 = 200
NOISE_REPEATS_H2 = 3

random.seed(SEED); torch.manual_seed(SEED)
loader_h2 = make_loader(dataset, batch_size=1, shuffle=True)

emb_specs = build_emb_enc_specs()
fusion_specs = build_fusion_specs()
all_specs_h2 = emb_specs + fusion_specs
all_keys_h2 = [f"{s.stage}_{s.component}" for s in all_specs_h2]

corrupt_conditions = ["context", "question", "both"]
h2_te = {cc: [] for cc in corrupt_conditions}
h2_ie = {cc: {k: [] for k in all_keys_h2} for cc in corrupt_conditions}

samples_h2 = 0
with torch.no_grad():
    for batch_idx, (Cwid, Ccid, Qwid, Qcid, y1, y2, ids) in enumerate(loader_h2):
        if samples_h2 >= NUM_SAMPLES_H2:
            break
        Cwid, Ccid = Cwid.to(DEVICE), Ccid.to(DEVICE)
        Qwid, Qcid = Qwid.to(DEVICE), Qcid.to(DEVICE)
        y1, y2 = y1.to(DEVICE), y2.to(DEVICE)

        p1_c, p2_c, clean_acts, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid, collect=True)
        prob_clean = compute_span_prob(p1_c, p2_c, y1, y2)
        if prob_clean.item() < MIN_CLEAN_PROB:
            continue

        for cc in corrupt_conditions:
            reps_te, reps_ie = [], {k: [] for k in all_keys_h2}
            for rep in range(NOISE_REPEATS_H2):
                ns = SEED + batch_idx * 100 + rep
                p1_x, p2_x, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid,
                                                  corrupt_target=cc, noise_std_scale=NOISE_STD_SCALE, noise_seed=ns)
                prob_x = compute_span_prob(p1_x, p2_x, y1, y2)
                te = (prob_clean - prob_x).item()
                reps_te.append(te)
                if abs(te) < 1e-6:
                    continue

                for spec, key in zip(all_specs_h2, all_keys_h2):
                    if cc == "context" and spec.stage == "emb_enc_Q":
                        continue
                    if cc == "question" and spec.stage == "emb_enc_C":
                        continue
                    p1_r, p2_r, _, _ = qanet_forward(model, Cwid, Ccid, Qwid, Qcid,
                                                      corrupt_target=cc, noise_std_scale=NOISE_STD_SCALE, noise_seed=ns,
                                                      clean_acts=clean_acts, restore_spec=spec)
                    prob_r = compute_span_prob(p1_r, p2_r, y1, y2)
                    reps_ie[key].append((prob_r - prob_x).item())

            h2_te[cc].append(np.mean(reps_te) if reps_te else 0.0)
            for key in all_keys_h2:
                if reps_ie[key]:
                    h2_ie[cc][key].append(np.mean(reps_ie[key]))

        samples_h2 += 1
        if samples_h2 % 50 == 0:
            print(f"  [{samples_h2}/{NUM_SAMPLES_H2}]  TE(C)={np.mean(h2_te['context']):.4f}  TE(Q)={np.mean(h2_te['question']):.4f}  TE(CQ)={np.mean(h2_te['both']):.4f}")

# Build results dict
h2_results = {}
for cc in corrupt_conditions:
    te_arr = np.array(h2_te[cc])
    n_te = len(te_arr)
    h2_results[cc] = {
        "te_mean": float(te_arr.mean()) if n_te else 0,
        "te_ci95": float(te_arr.std()*1.96/np.sqrt(n_te)) if n_te>1 else 0,
        "ie": {},
    }
    for key in all_keys_h2:
        vals = np.array(h2_ie[cc].get(key, []))
        n = len(vals)
        if n > 0:
            h2_results[cc]["ie"][key] = {
                "aie": float(vals.mean()),
                "ci95": float(vals.std()*1.96/np.sqrt(n)) if n>1 else 0,
                "nie": float(vals.mean() / max(abs(te_arr.mean()), 1e-8)),
            }

os.makedirs("experiments/results/H2", exist_ok=True)
with open("experiments/results/H2/h2_results.json", "w") as f:
    json.dump(h2_results, f, indent=2)

print(f"\nH2 done. {samples_h2} samples.")
for cc in corrupt_conditions:
    print(f"  {cc:>10s}:  TE = {h2_results[cc]['te_mean']:.4f} ± {h2_results[cc]['te_ci95']:.4f}")
    cq_ie = h2_results[cc]["ie"].get("cq_att_output", {})
    if cq_ie:
        print(f"{'':>13s} CQ-Att recovery = {cq_ie['nie']:.1%}")

In [ ]:
## H2 — Visualization
import matplotlib.pyplot as plt

# --- 1. TE comparison across corruption targets ---
fig, ax = plt.subplots(figsize=(6, 5))
cc_labels = ["Context", "Question", "Both"]
te_vals = [h2_results[cc]["te_mean"] for cc in corrupt_conditions]
te_errs = [h2_results[cc]["te_ci95"] for cc in corrupt_conditions]
colors_te = ["#2196F3", "#FF9800", "#F44336"]
ax.bar(cc_labels, te_vals, yerr=te_errs, color=colors_te, capsize=5, edgecolor="black")
ax.set_ylabel("Total Effect (TE)"); ax.set_title("H2: TE by Corruption Target")
fig.tight_layout(); plt.show()

# --- 2. NIE bars for each condition ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for i, cc in enumerate(corrupt_conditions):
    ax = axes[i]
    ie_dict = h2_results[cc]["ie"]
    keys = sorted(ie_dict.keys())
    vals = [ie_dict[k]["nie"] for k in keys]
    errs = [ie_dict[k]["ci95"] / max(abs(h2_results[cc]["te_mean"]),1e-8) for k in keys]
    labels = [k.replace("emb_enc_C_", "EncC:").replace("emb_enc_Q_", "EncQ:")
               .replace("cq_att_", "CQAtt:").replace("cq_resizer_", "Resize:") for k in keys]
    c = "#2196F3" if cc=="context" else ("#FF9800" if cc=="question" else "#F44336")
    ax.barh(range(len(keys)), vals, xerr=errs, color=c, alpha=0.8, capsize=3)
    ax.set_yticks(range(len(keys))); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Normalized IE"); ax.set_title(f"Corrupt: {cc_labels[i]}")
    ax.axvline(0, color="gray", ls="--", alpha=0.5)
fig.suptitle("H2: Normalized Indirect Effect by Restoration Point", fontsize=14)
fig.tight_layout(); plt.show()

# --- 3. CQ Attention bottleneck summary ---
print("\n=== CQ Attention Bottleneck Summary ===")
for cc in corrupt_conditions:
    ie_dict = h2_results[cc]["ie"]
    cq_nie = ie_dict.get("cq_att_output", {}).get("nie", 0)
    total_nie = sum(v["nie"] for v in ie_dict.values()) if ie_dict else 1
    print(f"  {cc:>10s}:  CQ-Att NIE = {cq_nie:.4f}  ({cq_nie/max(total_nie,1e-6):.1%} of total sum)")

### H3: Pointer Layer Asymmetric Connections & Pass Role Differentiation

**Hypothesis**: The asymmetric M1/M2 → start, M1/M3 → end is better than all symmetric alternatives. M2 and M3 encode distinct temporal features (M2 specializes in start-boundary, M3 in end-boundary).

**Method**: (A) Replace M1/M2/M3 wiring at the Pointer layer and measure F1/EM. (B) CKA & cosine similarity analysis.

In [ ]:
## H3 Phase A — Pointer Replacement Experiments
from experiments.run_H3 import POINTER_CONFIGS, pointer_forward, run_phase_a, run_phase_b
from experiments.run_H3 import cosine_sim_per_token, linear_cka
from EvaluateTools.eval_utils import convert_tokens, squad_evaluate

print("Phase A: Evaluating all pointer wiring configs on full dev set...\n")
phase_a = run_phase_a(model, dataset, eval_file, batch_size=32)

os.makedirs("experiments/results/H3", exist_ok=True)

print("\n=== Phase A Summary ===")
print(f"{'Config':>12s}  {'F1':>7s}  {'EM':>7s}  {'ΔF1':>7s}  {'ΔEM':>7s}")
print("-" * 50)
orig_f1 = phase_a["original"]["f1"]
orig_em = phase_a["original"]["em"]
for cfg, res in sorted(phase_a.items(), key=lambda x: -x[1]["f1"]):
    df1 = res["f1"] - orig_f1
    dem = res["em"] - orig_em
    marker = " ◀ baseline" if cfg == "original" else ""
    print(f"{cfg:>12s}  {res['f1']:7.2f}  {res['em']:7.2f}  {df1:+7.2f}  {dem:+7.2f}{marker}")

In [ ]:
## H3 Phase B — Representation Similarity Analysis
print("Phase B: Computing M1/M2/M3 representation similarity...\n")
phase_b = run_phase_b(model, dataset, num_samples=1000, batch_size=32, seed=42)

# Save all H3 results
with open("experiments/results/H3/h3_results.json", "w") as f:
    json.dump({"phase_a": phase_a, "phase_b": phase_b}, f, indent=2)

print("--- Global Cosine Similarity ---")
for pair, stats in phase_b["global_cosine"].items():
    print(f"  {pair}: {stats['mean']:.4f} ± {stats['ci95']:.4f}")

print("\n--- M2 vs M3 Cosine by Position ---")
for region, stats in phase_b["position_cosine_M2_M3"].items():
    print(f"  {region:>20s}: {stats['mean']:.4f} ± {stats['ci95']:.4f}")

print("\n--- CKA ---")
for pair in ["M1_M2", "M1_M3", "M2_M3"]:
    if pair in phase_b.get("cka", {}):
        print(f"  CKA({pair}) = {phase_b['cka'][pair]:.4f}")

In [ ]:
## H3 — Visualization
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 1. Phase A: F1/EM bar chart ---
ax = axes[0]
cfgs = sorted(phase_a.keys(), key=lambda c: -phase_a[c]["f1"])
f1s = [phase_a[c]["f1"] for c in cfgs]
ems = [phase_a[c]["em"] for c in cfgs]
x = np.arange(len(cfgs))
w = 0.35
ax.bar(x - w/2, f1s, w, label="F1", color="#2196F3", edgecolor="black")
ax.bar(x + w/2, ems, w, label="EM", color="#FF9800", edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(cfgs, rotation=30, ha="right")
ax.set_ylabel("Score"); ax.set_title("H3-A: Pointer Wiring Configs")
ax.legend(); ax.axhline(phase_a["original"]["f1"], color="blue", ls="--", alpha=0.3)

# --- 2. Phase B: CKA matrix ---
ax = axes[1]
pairs_order = ["M1", "M2", "M3"]
cka_mat = np.eye(3)
cka = phase_b.get("cka", {})
pair_map = {("M1","M2"): cka.get("M1_M2", 0), ("M1","M3"): cka.get("M1_M3", 0), ("M2","M3"): cka.get("M2_M3", 0)}
for (a,b), v in pair_map.items():
    i, j = pairs_order.index(a), pairs_order.index(b)
    cka_mat[i,j] = cka_mat[j,i] = v
sns.heatmap(cka_mat, xticklabels=pairs_order, yticklabels=pairs_order,
            annot=True, fmt=".3f", cmap="YlOrRd", vmin=0, vmax=1, ax=ax)
ax.set_title("H3-B: CKA Similarity")

# --- 3. Phase B: Position-stratified cosine ---
ax = axes[2]
pos_data = phase_b.get("position_cosine_M2_M3", {})
regions = ["answer_start", "answer_interior", "answer_end", "non_answer"]
means_pos = [pos_data.get(r, {}).get("mean", 0) for r in regions]
errs_pos = [pos_data.get(r, {}).get("ci95", 0) for r in regions]
colors_pos = ["#4CAF50", "#8BC34A", "#FF9800", "#9E9E9E"]
ax.bar(["Start", "Interior", "End", "Non-answer"], means_pos, yerr=errs_pos,
       color=colors_pos, capsize=5, edgecolor="black")
ax.set_ylabel("Cosine Sim (M2 vs M3)")
ax.set_title("H3-B: M2 vs M3 by Position")

fig.suptitle("H3: Pointer Asymmetry & Pass Role Differentiation", fontsize=14, fontweight="bold")
fig.tight_layout(); plt.show()

### Stage 3 — Results Summary

All results are saved to `experiments/results/`:
- `H1/h1_results.json` — Component-level causal tracing AIE values
- `H2/h2_results.json` — Dual-stream TE & NIE per corruption target
- `H3/h3_results.json` — Pointer replacement F1/EM & representation similarity

In [ ]:
## Stage 3 — Summary printout
print("=" * 60)
print("STAGE 3 RESULTS SUMMARY")
print("=" * 60)

print("\n[H1] Component-Level Causal Tracing")
print(f"  Samples: {h1_results[list(h1_results.keys())[0]]['n']}")
top3 = sorted(h1_results.items(), key=lambda x: -x[1]["aie_span"])[:3]
for k, v in top3:
    print(f"  Top: {k:<25s}  AIE = {v['aie_span']:.4f}")

print(f"\n[H2] Dual-Stream Information Flow")
for cc in ["context", "question", "both"]:
    te = h2_results[cc]["te_mean"]
    cq = h2_results[cc]["ie"].get("cq_att_output", {}).get("nie", 0)
    print(f"  {cc:>10s}: TE = {te:.4f}, CQ-Att recovery = {cq:.1%}")

print(f"\n[H3] Pointer Asymmetry")
for cfg in ["original", "swap", "sym_M2", "sym_M3"]:
    if cfg in phase_a:
        r = phase_a[cfg]
        print(f"  {cfg:>12s}: F1 = {r['f1']:.2f}, EM = {r['em']:.2f}")
print(f"  CKA(M2,M3) = {phase_b.get('cka',{}).get('M2_M3', '?')}")

print("\n" + "=" * 60)
print("All results saved to experiments/results/")
print("=" * 60)